# Getting Data 

Data Science is all about acquiring, cleaning, and transforming data. Getting data into Python and formatting it properly:

## stdin and stdout

When running Python scripts in the command line, you can *pipe* data through them using ```sys.stdin``` and ```sys.stdout```. For example, script that reads in line of text and spits back out the ones that match a regular expression. 

In [ ]:
import sys, re

# sys.argv is the list of command-line arguments
# sys.argv[0] is the name of the program itself
# sys.argv[1] will be the regex specified at the command line
regex = sys.argv[1]

# for every line passed into the script
for line in sys.stdin:
    # if it matches the regex, write it to stdout
    if re.search(regex, line):
        sys.stdout.write(line)

The one that counts the lines it receieves and then writes out the count:


In [ ]:
# line_count.py
import sys 

count = 0
for line in sys.stdin:
    count += 1
    
# print goes to sys.stdout
print(count)
    

KeyboardInterrupt: 

You could then use them to count how many lines of a file contain numbers. In Windows:

```
type SomeFile.txt | python egrep.py "[0-9]" | python line_count.py
```

The | is the pipe character, which means "use the output of the left command as the input of the right command." 


Here's a script that counts the words in its input and writes out the most common ones:

In [ ]:
# most_common_words.py
import sys
from collections import Counter 

# pass in number of words as first argument
try: 
    num_words = int(sys.argv[1])
except: 
    print("usage: most_common_words.py num_words")
    sys.exit(1) # nonzero exit code indicates error
    
counter = Counter(word.lower() # lowercase words
                for line in sys.stdin # split on spaces
                for word in line.strip().split() if word) # skip empty 'words'

for word, count in counter.most_common(num_words):
    sys.stdout.write(str(count))
    sys.stdout.write("\t")
    sys.stdout.write(word)
    sys.stdout.write("\n")

After which you could do something like:

```
$ cat the_bible.txt | python most_common_words.py 10
36397 the
30031 and
20163 of
7154 to
6484 in
5856 that
5421 he
5226 his
5060 unto
4297 shall
```

## Reading Files

You can also explicitly read from and write to files directly in code. Python makes working with files pretty simple. 

### The Basics of Text Files

The first step to working with a text file is to obtain a *file* object using ```open```:


In [ ]:
# 'r' means read-only, it's assumed  if you leave it out
file_for_reading = open('reading_file.txt', 'r')
file_for_reading2 = open('reading_file.txt')

# 'w' is write -- will destroy the file if it already exists!
file_for_writing = open('writing_file.txt', 'w')

# 'a' is append -- for adding to the end of the file
file_for_appending = open('appending_file.txt', 'a')

# don't forget to close your files when you're done
file_for_writing.close()

FileNotFoundError: [Errno 2] No such file or directory: 'reading_file.txt'

Because it is easy to forget to close your files, you should always use them in a ```with``` block, at the end of which they will be closed automatically:

In [ ]:
with open(filename) as f:
    data = function_that_gets_data(f)
    
# at this point f has already been closed, so don't try to use it
process(data)


If you need to read a whole text file, you can just iterate over the lines of the file using ```for```:


In [ ]:
starts_with_hash = 0

with open('input.txt') as f:
    for line in f:                  # look at each line in the file
        if re.match("^#", line):    # use a regex to see if it start with '#'
            starts_with_hash += 1   # if it does, add 1 to the count

Every line you get this way ends in a newline character, so you'll often want to ```strip``` it before doing anything with it. 

For example, imagine you have a file full of email addresses, one per line, and you need to generate a histogram of the domains. A good first approximation is to just take the parts of the email addresses that come after the @. 

In [ ]:
def get_domain(email_address: str) -> str:
    """Split on '@' and return the last piece"""
    return email_address.lower().split("@")[-1]

# a couple of tests
assert get_domain('joelgrus@gmail.com') == 'gmail.com'
assert get_domain('joel@m.datasciencester.com') == 'm.datasciencester.com'

from collections import Counter

with open('email_addresses.txt', 'r') as f:
    domain_counts = Counter(get_domain(line.strip())
        for line in f
        if "@" in line)

FileNotFoundError: [Errno 2] No such file or directory: 'email_addresses.txt'

### Delimited Files

More frequently you'll work with files that have lots of data on each line — very often either **comma-separated** or **tab-separated**. Each line has several fields, with a comma or tab indicating where one field ends and the next begins.

> ⚠️ **Warning:** Never parse a comma-separated file yourself. You will screw up the edge cases! Always use Python's `csv` module, `pandas`, or another library designed for this purpose.

#### Reading Files Without Headers

If your file has no headers, use `csv.reader` to iterate over rows — each row will be an appropriately split list:

```
6/20/2014	AAPL	90.91
6/20/2014	MSFT	41.68
6/20/2014	FB  	64.5
6/19/2014	AAPL	91.86
6/19/2014	MSFT	41.51
6/19/2014	FB  	64.34
```

we could process them with:

In [ ]:
import csv
with open('tab_delimited_stock_price.txt') as f:
    tab_reader = csv.reader(f, delimiter='\t')
    for row in tab_reader:
        date = row[0]
        symbol = row[1]
        closing_price = float(row[2])
        process(data, symbol, closing_price)


If your file has headers:

```
date:symbol:closing_price
6/20/2014:AAPL:90.91
6/20/2014:MSFT:41.68
6/20/2014:FB:64.5
```

You can either skip the header row manually or use `csv.DictReader` to get each row as a dictionary with the headers as keys:


In [ ]:
import csv 
with open('colo_deliminited_stock_prices.txt') as f:
    colon_reader = csv.DictReader(f, delimiter=':')
    date = dict_row["date"]
    symbol = dict_row["symbol"]
    closing_price = float(dict_row["closing_price"])
    process(date, symbol, closing_price)

SyntaxError: invalid syntax (3336449950.py, line 3)

## Scraping the Web

### HTML and the Parsing Thereof

Web pages are written in HTML, where text is marked up into elements and attributes:

```html
<html>
  <head><title>Joel Grus</title></head>
  <body>
    <p id="subject">Data Science</p>
  </body>
</html>
```

In a perfect world we could extract data with simple rules like *"find the `<p>` element whose id is `subject` and return its text."* In practice, HTML is rarely well-formed or consistently annotated — so we need help parsing it.

### Beautiful Soup

**Beautiful Soup** builds a tree out of the elements on a web page and provides a simple interface for accessing them. We'll also use:

- **Requests** — a nicer way to make HTTP requests than Python's built-ins
- **html5lib** — a more lenient HTML parser that handles malformed HTML gracefully

Install all three:

```
python -m pip install beautifulsoup4 requests html5lib
```

To use Beatiful Soup, we pass a string containing HTML into the ```BeautifulSoap``` function. Here this will be the result of a call to ```requests.get```:

In [ ]:
from bs4 import BeautifulSoup
import requests 

url = ("https://raw.githubusercontent.com/"
"joelgrus/data/master/getting-data.html")
html = requests.get(url).text
soup = BeautifulSoup(html, 'html5lib')

after which we can get pretty far using a few simple methods. We’ll typically work with Tag objects, which correspond to the tags representing the structure of an HTML page.


For example, to find the first ```<p>``` tag (and its contents), you can use:

In [ ]:
first_paragraph = soup.find('p') # or just soup.p

You can get the text contents of a ```Tag``` using its ```text``` property:

In [ ]:
first_paragraph_text = soup.p.text
first_paragraph_words = soup.p.text.split()

And you can extract a tag’s attributes by treating it like a dict:

In [ ]:
first_paragraph_id = soup.p['id'] # raises KeyError if no 'id'
first_paragraph_id2 = soup.p.get('id') # returns None if no 'id'

You can get multiple tags at once as follows:

In [ ]:
all_paragraphs = soup.find_all('p') # or just soup('p')
paragraphs_with_ids = [p for p in soup('p') if p.get('id')]

Frequently, you’ll want to find tags with a specific class:

In [ ]:
important_paragraphs = soup('p', {'class' : 'important'})
important_paragraphs2 = soup('p', 'important')
important_paragraphs3 = [p for p in soup('p')
                        if 'important' in p.get('class', [])]

And you can combine these methods to implement more elaborate logic.
For example, if you want to find every ```<span>``` element that is contained
inside a ```<div>``` element, you could do this:

In [ ]:
# Warning: will return the same <span> multiple times
# if it sits inside multiple <div>s.
# Be more clever if that's the case.
spans_inside_divs = [span
for div in soup('div') # for each <div> on the page
for span in div('span')] # find each <span> inside it